# Step 09 — projecting the later visits *(optional)*

Six donors were sampled more than once. Step 00 kept only the first visit, because every correlation
in WGCNA assumes independent samples. The later visits were not discarded — they were set aside for
this.

They support a question the cross-sectional analysis cannot ask. **Disease activity falls across the
visits** (D256: SLEDAI-2K 9 → 6 → 4 → 2; D173: 12 → 5; D95: 10 → 10 → 10 → 8 → 8), so if a module
tracks disease activity, its eigenprotein should fall in the same person, on axes those samples never
influenced.

This is a small n — ten samples from six people — so it is a **sanity check, not evidence**. It can
falsify a module's claim to track activity; it cannot establish one.

In [ ]:
suppressMessages(library(WGCNA))
w   <- readRDS("artifacts/wgcna_A.rds"); X <- w$X; mods <- w$mods
TP  <- read.csv("R_timepoints_log2_combat.csv", row.names = 1, check.names = FALSE)
tpm <- read.csv("R_timepoints_meta.csv",        row.names = 1, check.names = FALSE)
slem<- read.csv("R_cohort-A_meta.csv", row.names = 1, check.names = FALSE)
TP  <- TP[, colnames(X), drop = FALSE]
tpm[, c("DonorId","Visit","SLEDAI_2K","Disease_activity")]

In [ ]:
# Project samples onto modules that were defined WITHOUT them.
#
# Two things make this a projection rather than a second fit, and both are easy
# to get wrong in a way that silently destroys the answer:
#
#  1. the loadings come from the SLE matrix alone and are never recomputed;
#  2. the new samples are scaled with SLE's per-protein mean and sd, NOT their
#     own. Re-centring on the new samples would move their mean to zero, which
#     is precisely the difference being measured.
project_module <- function(md0, X, mods, NEW) {
  g  <- colnames(X)[mods == md0]
  mu <- colMeans(X[, g, drop = FALSE])
  sdv<- apply(X[, g, drop = FALSE], 2, sd)
  keep <- sdv > 0
  g <- g[keep]; mu <- mu[keep]; sdv <- sdv[keep]

  Zs <- scale(X[, g, drop = FALSE], center = mu, scale = sdv)
  v  <- svd(Zs, nu = 0, nv = 1)$v[, 1]
  e  <- as.vector(Zs %*% v)
  # WGCNA aligns an eigenprotein so it tracks the module's mean abundance.
  if (cor(e, rowMeans(Zs)) < 0) { v <- -v; e <- -e }

  Zn <- scale(as.matrix(NEW[, g, drop = FALSE]), center = mu, scale = sdv)
  list(sle = e, new = as.vector(Zn %*% v))
}

In [ ]:
mlist <- setdiff(unique(mods), "grey")
ME <- moduleEigengenes(X, mods)$eigengenes
# named X would bind to lapply's own first formal -- see step 08
P  <- lapply(mlist, function(k) project_module(k, X, mods, TP)); names(P) <- mlist

gate <- sapply(mlist, function(k) abs(cor(P[[k]]$sle, ME[[paste0("ME", k)]])))
stopifnot(min(gate) > 0.99)
cat(sprintf("GATE PASSED (min round-trip r = %.4f)\n", min(gate)))

## Does the eigenprotein fall when activity falls?

Only donors whose first visit is in **cohort A** can be compared, since that is the cohort the
modules were fitted on. For each such donor, pair the first visit's own eigenprotein with each later
visit's projected one, and correlate the change in eigenprotein against the change in SLEDAI-2K
across all pairs.

In [ ]:
ifn <- mods[grep("14148", colnames(X))[1]]
inA <- intersect(tpm$DonorId, slem$DonorId)
cat("repeat donors whose first visit is in cohort A:", length(inA), "\n")

if (length(inA) == 0) {
  cat("None -- the frozen split put every repeat donor's first visit in B or C.\n",
      "Re-run this notebook against whichever cohort holds them, or fit on all 260.\n")
} else {
  rows <- do.call(rbind, lapply(mlist, function(k) {
    first <- setNames(P[[k]]$sle, rownames(X))
    later <- setNames(P[[k]]$new, rownames(TP))
    d <- do.call(rbind, lapply(which(tpm$DonorId %in% inA), function(i) {
      id <- rownames(slem)[slem$DonorId == tpm$DonorId[i]][1]
      data.frame(d_eigen = later[rownames(TP)[i]] - first[[id]],
                 d_sledai = tpm$SLEDAI_2K[i] - slem[id, "SLEDAI_2K"])
    }))
    data.frame(module = k, n_pairs = nrow(d),
               r = round(suppressWarnings(cor(d$d_eigen, d$d_sledai,
                                              use = "complete.obs")), 2))
  }))
  rows <- rows[order(-abs(rows$r)), ]
  print(head(rows, 10)); print(rows[rows$module == ifn, ])
}

Whatever this shows, the honest framing is the same: with a handful of pairs, a correlation near 1
is unremarkable and a correlation near 0 is uninformative. What would be *informative* is a module
that moves confidently in the **wrong** direction — which would argue against its cross-sectional
association with activity.